# Análise Exploratória e Decisões de Modelagem (dbt)

Este notebook consolida a análise inicial sobre os dados brutos armazenados no Google Cloud Storage, visando entender seus perfis, anomalias e justificar como essas descobertas embasaram a estrutura dos modelos Staging no projeto dbt.

Estaremos acessando diretamente os parquets no GCS usando DuckDB.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

### 1. Tabela Escola
Analisando a estrutura original em contraposição à documentação.

In [ ]:
con.execute("""
SELECT * FROM read_parquet('https://storage.googleapis.com/case_vagas/rmi/escola') LIMIT 5
""").df()

**Descoberta & Decisão:**
Notamos que as colunas documentadas `tipo` e `regiao` não estão presentes fisicamente no arquivo parquet (apenas `id_escola` e `bairro` existem). Para evitar que o pipeline falhe duramente e garantir que a estrutura da camada de apresentação contemple o contrato de dados esperado, o modelo `stg_educacao__escola` foi ajustado para projetar essas colunas faltantes como nulas (`cast(null as varchar) as tipo`, etc).

### 2. Tabela Avaliação
Análise do formato de gravação das notas.

In [ ]:
con.execute("""
SELECT * FROM read_parquet('https://storage.googleapis.com/case_vagas/rmi/avaliacao') LIMIT 5
""").df()

**Descoberta & Decisão:**
O arquivo Parquet apresenta uma estrutura larga (wide) para as disciplinas (`disciplina_1` até `disciplina_4`), diferente de um formato longo (long) com uma coluna consolidada `nota`. O `stg_educacao__avaliacao` foi construído respeitando essa forma (importando explicitamente cada coluna de disciplina e a métrica agrupada `frequencia`), aplicando expectativas do pacote `dbt_expectations` sobre os arrays de limites para validar cada nota individual.

### 3. Tabela Frequência
Analisando a granularidade.

In [ ]:
con.execute("""
SELECT * FROM read_parquet('https://storage.googleapis.com/case_vagas/rmi/frequencia') LIMIT 5
""").df()

**Descoberta & Decisão:**
Apesar do contexto sugerir marcações diárias com `status` de presença, os dados reais representam fatias de tempo/consolidado (colunas `data_inicio`, `frequencia` em taxa decimal e `disciplina`). Ajustamos o staging e as tabelas intermediárias (`int_educacao__aluno_frequencia`) para utilizar a taxa média (`avg(frequencia)`) na construção dos cálculos das agregações de absenteísmo, providenciando integridade da regra de negócio final independente das anomalias nas granularidades diárias.